# Session 2: Git and reproducible research code
**120 minutes, including a 10-minute break.**

You will turn an expectation into a test, commit a small change on a branch,
resolve a deliberate conflict and explain what a reviewer should inspect.
Run shell blocks in a terminal; notebook cells remain read-only demonstrations.
Use Git Bash for shell blocks on Windows.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.table import QTable
# The Hub pulls the course into your workspace before starting this notebook.
# Prefer that editable checkout; the local editable install is a fallback.
import sys
_candidates = [Path.cwd(), *Path.cwd().parents, Path.home() / "jupyter_data_languages"]
for _candidate in _candidates:
    if (_candidate / "course2026/__init__.py").is_file() and (_candidate / "data/mock_galaxies.ecsv").is_file():
        sys.path.insert(0, str(_candidate))
        break
import course2026
ROOT = Path(course2026.__file__).resolve().parents[1]
plt.rcParams.update({'figure.figsize': (7, 4), 'font.size': 11})

## 0–10 min: from notebook to a function
A notebook is useful for exploration. A reusable module gives a calculation an
interface, makes tests easy to rerun and removes hidden cell-order dependencies.
This reference helper packages the Session 1 mask as a function; compare the
two calculations line by line.


In [ ]:
import inspect
from course2026.analysis import usable_rows
print(inspect.getsource(usable_rows))
table = QTable.read(ROOT / 'data/mock_galaxies.ecsv')
assert usable_rows(table).sum() == 395

## 10–40 min: a local Git practical
From the course root, create an independent exercise repository. This script
refuses to overwrite an existing directory. Keep the terminal in this sandbox
for the remainder of the Git practical.

```bash
python scripts/create_practice_repo.py outputs/git-practice
cd outputs/git-practice
git config user.name "Your name"
git config user.email "your-address@example.org"
python -m pytest -q
git status
git add angle.py test_angle.py README.md .gitignore
git diff --staged
git commit -m "Add tested bearing separation helper"
git log --oneline
```

The identity settings apply only to this sandbox. Use your own address or your
Git hosting provider's no-reply address. No remote is configured.

### Task A — ten minutes
Create a branch with `git switch -c test-wraparound`. Add these to `test_angle.py`:

```python
def test_wraparound():
    assert angle_distance(1, 359) == 2

def test_full_turns():
    assert angle_distance(720, 90) == 90
```

Run `python -m pytest -q`, inspect `git diff`, then stage and commit just the test
file. Explain the distinction between the working tree, staging area and commit.
`git restore --staged test_angle.py` unstages a change without discarding edits.
Use `git diff --staged` to inspect exactly what the next commit will contain.

## 40–60 min: merge and resolve a conflict
First integrate the passing test branch:

```bash
git switch main
git merge test-wraparound
git switch -c description-a
```

Edit the first paragraph of `README.md` to say `Angles are measured in degrees.`
Then:

```bash
git add README.md
git commit -m "Describe angle units"
git switch main
git switch -c description-b
```

Edit that **same original paragraph** to say `The helper returns the shortest arc.`
Commit it, then bring both edits together:

```bash
git add README.md
git commit -m "Describe the returned separation"
git switch main
git merge description-a
git merge description-b
```

The final merge should conflict. Run `git status`, open `README.md`, and replace
all conflict markers with a sentence retaining **both** facts. Then:

```bash
git add README.md
git commit -m "Explain units and shortest-arc behavior"
python -m pytest -q
git log --oneline --graph --all
```

If you need to return to the pre-merge state while the conflict is unresolved,
`git merge --abort` does that. Discuss the intended scientific meaning before
choosing which version to retain. **60–70 min: break.**

## 70–95 min: tests need an independent expectation
If you opened a new terminal, return to `outputs/git-practice`. Check your location
before changing a branch: `git rev-parse --show-toplevel` must end in that sandbox.
Activate the course environment first if this is a new local terminal.

Create a fresh branch for the next review:

```bash
git switch -c edge-case-tests
```

A test that repeats the implementation can repeat its mistake. Use a geometric
answer: the separation between 1° and 359° is 2°. Add equal-angle and exactly
180° cases in your sandbox. Use `pytest.approx` for floating-point calculations;
choose tolerances from the problem's scale and expected numerical error.

This cell runs the course reference checks in a separate process. It checks the
actual helper that reads the catalogue, not just a notebook copy.


In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pytest', '-q',
    str(ROOT / 'tests/test_analysis.py')], cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
assert result.returncode == 0, result.stderr

### Task B — ten minutes
Temporarily replace the sandbox's angle formula with `abs(b - a)`. Predict which
tests will fail, run them, and read the failure report. Inspect `git diff -- angle.py`
to see the bug. Restore the correct formula in the editor and rerun. Do **not**
change expected answers to make a bug pass.

The formula diff should now be empty. Commit the new tests on `edge-case-tests`:

```bash
python -m pytest -q
git diff -- angle.py
git add test_angle.py
git diff --staged
git commit -m "Test equal and opposite bearings"
```

What does a passing test fail to establish? Examples: the input unit was wrong,
the measurements are correlated, or the model does not describe the population.


## 95–110 min: a reproducible handoff
Use the interpreter from the course environment. For a **local uv checkout**,
inspect packages with `uv pip freeze`; uv environments do not include pip by
default. In **CPD JupyterLab** or the standard pip installation, use
`python -m pip freeze`. Check `python -c "import sys; print(sys.executable)"`
if you are unsure which interpreter the terminal uses. Then, in your sandbox:

```bash
python -m pytest -q
```

`pyproject.toml` describes dependencies; `uv.lock` records a resolved environment.
An environment is not the same as a notebook kernel: check `sys.executable` if
imports differ. Restart the kernel and Run All before sharing a result.

A useful research repository has a README explaining the question, setup,
input provenance, commands, expected outputs and limits. Record the code revision,
configuration and random seed with results; a seed alone is not a complete
reproduction recipe. Keep `.venv`, credentials, generated outputs and private data
out of Git. `.gitignore` does not remove files already tracked.

### Optional hosting exercise: remotes and pull requests
Use an **empty** instructor-provided or personally owned classroom repository,
created without an initial README or other commit. Replace the URL placeholder
below with its actual URL before running the block. The earlier `test-wraparound`
branch is already merged; the new tests to review are on `edge-case-tests`.

```bash
git remote add origin YOUR_EMPTY_CLASSROOM_REPOSITORY_URL
git remote -v
git push -u origin main
git push -u origin edge-case-tests
git fetch origin
git diff origin/main...edge-case-tests
```

Pushing `main` establishes a shared base. Open a pull request from `edge-case-tests`
to `main` in **that classroom repository**. A partner reviews its description,
diff and tests before merging. An existing nonempty classroom repository needs
the instructor's clone workflow with shared history; it cannot use this independent
sandbox's history. Keep practice changes out of research upstreams.

If offline, `git show edge-case-tests` and `git diff main...edge-case-tests`
provide the same review conversation. Hosting is optional.


## 110–120 min: partner review and exit check
Show your partner `git show edge-case-tests`, its test diff and passing tests.
Explain why each expectation is independent of the implementation. Add one sentence
to the README about a limitation, then commit that reviewed change. Use `git status`
to check that the intended files are committed.

To undo a **committed** ordinary mistake without rewriting shared history, make a
new commit with `git revert --no-edit <commit-id>` and rerun the tests. Practise on
an ordinary sandbox commit, not the conflict merge. Secrets require credential
revocation and proper incident handling; reverting a commit does not erase it.

**Deliverable:** a passing test change on a branch, a resolved conflict and a
reviewed commit. Git records changes; tests check specified behavior; scientific
validation checks whether that behavior answers the research question.
